# 1. STAC Catalog Exploration — Microsoft Planetary Computer NAIP

This notebook demonstrates how `stac_client.py` works. It searches the Microsoft Planetary Computer STAC API for NAIP imagery matching a bounding box and date range, lists available COG assets, and finds the best overlapping image for a given polygon geometry.

**Key functions demonstrated:**
- `find_imagery_for_bbox(bbox, datetime_str)` — search by bounding box
- `find_best_imagery_for_polygon(polygon_geometry, datetime_str)` — find overlapping imagery for a GeoJSON polygon

## Imports & setup

In [ ]:
import json
import os
import sys

import numpy as np

# Add backend to path so we can import engine modules
sys.path.insert(0, os.path.join(os.getcwd(), "..", "backend"))

from edge_engine.stac_client import (
    STAC_API_URL,
)

print(f"STAC API URL: {STAC_API_URL}")
print("All imports successful.")

## 1.1 Define a bounding box

We'll use a small region in Iowa (a common NAIP coverage area). The bbox format is `[west, south, east, north]`.

In [ ]:
# Bounding box: a small farm patch in Iowa
bbox = [-93.75, 41.55, -93.70, 41.58]
datetime_range = "2020-01-01/2021-12-31"

print(f"Bounding box: {bbox}")
print(f"Date range:   {datetime_range}")
print("Collection:   naip")

## 1.2 Search for NAIP imagery

`find_imagery_for_bbox` opens the STAC catalog, runs a search, and returns a list of records with `id`, `bbox`, `geometry`, `datetime`, `cog_url`, and `gsd`.

> **Note:** This cell mocks the STAC response so the notebook can run offline. In production the real API call is used.

In [ ]:
def mock_find_imagery_for_bbox(bbox, datetime_str=None):
    """Return synthetic NAIP records identical in shape to real API responses."""
    np.random.seed(42)
    records = []
    for i in range(3):
        cog_url = (
            f"https://naipblobs.blob.core.windows.net/naip/v002/"
            f"ia/2020/ia_100cm_2020_{i:03d}/"
            f"{i:04d}300{chr(97 + i)}.tif"
        )
        records.append(
            {
                "id": f"NAIP_IA_2020_{i:03d}",
                "bbox": [
                    bbox[0] + i * 0.01,
                    bbox[1] + i * 0.01,
                    bbox[2] + i * 0.01,
                    bbox[3] + i * 0.01,
                ],
                "geometry": {
                    "type": "Polygon",
                    "coordinates": [
                        [
                            [bbox[0] + i * 0.01, bbox[1] + i * 0.01],
                            [bbox[2] + i * 0.01, bbox[1] + i * 0.01],
                            [bbox[2] + i * 0.01, bbox[3] + i * 0.01],
                            [bbox[0] + i * 0.01, bbox[3] + i * 0.01],
                            [bbox[0] + i * 0.01, bbox[1] + i * 0.01],
                        ]
                    ],
                },
                "datetime": f"2020-07-{15 + i:02d}T17:00:00Z",
                "cog_url": cog_url,
                "gsd": round(np.random.uniform(0.6, 1.0), 2),
            }
        )
    return records


records = mock_find_imagery_for_bbox(bbox, datetime_range)
print(f"Found {len(records)} imagery record(s):\n")
for r in records:
    print(f"  ID:       {r['id']}")
    print(f"  Date:     {r['datetime']}")
    print(f"  GSD:      {r['gsd']} m")
    print(f"  COG URL:  {r['cog_url']}")
    print(f"  BBOX:     {r['bbox']}")
    print()

## 1.3 Explore available assets

Each record exposes a `cog_url` pointing to a Cloud-Optimized GeoTIFF. We can inspect the URL pattern and the bounding box geometry.

In [ ]:
# Pick the first record and display its details
best = records[0]
print("Selected imagery:")
print(json.dumps(best, indent=2))

## 1.4 Find overlapping imagery for a polygon

`find_best_imagery_for_polygon` takes a GeoJSON geometry, computes its bounding box, calls `find_imagery_for_bbox`, and returns the first candidate whose bbox intersects the polygon.

We'll use a small polygon inside our search area.

In [ ]:
polygon_geometry = {
    "type": "Polygon",
    "coordinates": [
        [
            [-93.74, 41.56],
            [-93.72, 41.56],
            [-93.72, 41.57],
            [-93.74, 41.57],
            [-93.74, 41.56],
        ]
    ],
}

print("Polygon geometry:")
print(json.dumps(polygon_geometry, indent=2))

In [ ]:
def mock_find_best_imagery_for_polygon(polygon_geometry, datetime_str=None):
    """Mock version that simulates the overlap logic."""
    from shapely.geometry import box, shape

    poly_shape = shape(polygon_geometry)
    bbox = list(poly_shape.bounds)
    candidates = mock_find_imagery_for_bbox(bbox, datetime_str)
    for c in candidates:
        candidate_geom = box(*c["bbox"])
        if candidate_geom.intersects(poly_shape):
            return c
    return candidates[0] if candidates else None


result = mock_find_best_imagery_for_polygon(polygon_geometry, datetime_range)
if result:
    print("Best overlapping imagery:")
    print(f"  ID:      {result['id']}")
    print(f"  COG URL: {result['cog_url']}")
else:
    print("No overlapping imagery found.")

## 1.5 How `stac_client.py` works under the hood

1. `STAC_API_URL` is read from the env var `STAC_API_URL`, defaulting to `https://planetarycomputer.microsoft.com/api/stac/v1`.
2. `find_imagery_for_bbox` opens a `pystac_client.Client`, searches the `naip` collection with the given bbox and datetime, and collects up to 5 items.
3. For each item it extracts the `image` asset — a COG URL — along with `id`, `bbox`, `geometry`, `datetime`, and `gsd`.
4. `find_best_imagery_for_polygon` derives a bbox from the polygon, delegates to `find_imagery_for_bbox`, then iterates candidates returning the first that intersects.

In [ ]:
print("stac_client.py source:")
with open("../backend/edge_engine/stac_client.py") as f:
    print(f.read())